# Nautilus chains: GCsp + WL (symbolic_new), 2 cosmological parameters + nuisances

A deliberately small end-to-end nested-sampling example:

* two free cosmological parameters (default `Omegam`, `sigma8`) with the
  **symbolic_new** (`code: "symbolic"`) backend for both Euclid probes,
* separable nuisance priors so the chains also sample the automatic nuisances
  (GCsp `lnbg_i`/`Ps_i`, photo bias, intrinsic alignment),
* one Nautilus chain per probe (`WL`, `GCsp`), Fisher forecasts for the same
  configuration, and **combined contour plots** of chains *and* Fishers via
  `cosmicfishpie.analysis.fishconsumer`,
* chain statistics: `fco.plot_chain_summary` (ChainConsumer errorbar summary),
  effective sample size, weighted mean/$\sigma$, chain-$\sigma$ vs Fisher-$\sigma$
  ratios and mean biases, plus the Fisher-level KL information gain.

Everything goes through the native `fco`/`cfop` helpers - no chainconsumer /
getdist code written by hand.  Chains and Fishers are stored under
`notebooks/results/`.

In [ ]:
%matplotlib inline

import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from nautilus import Sampler

from cosmicfishpie.fishermatrix.cosmicfish import FisherMatrix
from cosmicfishpie.likelihood.photo_like import PhotometricLikelihood
from cosmicfishpie.likelihood.spectro_like import SpectroLikelihood
import cosmicfishpie.analysis.fishconsumer as fco
import cosmicfishpie.analysis.fisher_operations as cfop

sns.set_theme(context="talk", style="ticks")

RESULTS = Path("./results"); RESULTS.mkdir(parents=True, exist_ok=True)
PLOTS = Path("./plots");     PLOTS.mkdir(parents=True, exist_ok=True)

# ---- run controls -------------------------------------------------------
FREE_PARAMS = ["Omegam", "sigma8"]        # the "2 cosmo parameters"
SAMPLE_NUISANCES = True                   # False -> nuisances fixed at fiducial
N_LIVE = 50                               # nested-sampling live points (small = fast demo)
N_WORKERS = 1                             # nautilus pool; e.g. nproc - 2

print("free:", FREE_PARAMS, "| sample nuisances:", SAMPLE_NUISANCES, "| n_live:", N_LIVE)

In [ ]:
# --- shared definitions (same pattern as
# scripts/likelihood/nautilus/wl_gcsp_fisher_nautilus_demo.py) -------------

FIDUCIAL = {
    "Omegam": 0.32, "Omegab": 0.05, "h": 0.67, "ns": 0.96, "sigma8": 0.815584,
}
PRIOR_RANGES = {
    "Omegam": (0.28, 0.36),
    "Omegab": (0.04, 0.06),
    "h": (0.60, 0.74),
    "ns": (0.90, 1.02),
    "sigma8": (0.75, 0.87),
}

COMMON_OPTIONS = {
    "accuracy": 1, "feedback": 1, "code": "symbolic", "nonlinear": True,
    "cosmo_model": "LCDM", "survey_name": "Euclid",
    "survey_name_photo": "Euclid-Photometric-ISTF-Pessimistic",
    "survey_name_spectro": "Euclid-Spectroscopic-ISTF-Pessimistic",
    "derivatives": "3PT",
    "results_dir": str(RESULTS) + "/",
}


def options_for(outroot):
    return {**COMMON_OPTIONS, "outroot": outroot}


def conditional_fisher(fisher, params):
    """Sub-block: all other parameters held FIXED (conditional)."""
    return cfop.reshuffle(fisher, params, update_names=False)


def marginal_fisher(fisher, params):
    """Schur complement: all other parameters integrated out (marginal)."""
    return cfop.marginalise(fisher, params, update_names=False)


def nuisance_prior_ranges(fisher, free_params, n_sigma=5.0):
    """Box priors: cosmology from PRIOR_RANGES, nuisances fid +/- n_sigma * sigma_marg."""
    ranges = {p: PRIOR_RANGES[p] for p in free_params}
    names = list(fisher.param_names)
    fiducials = np.asarray(fisher.param_fiducial, dtype=float)
    sigmas = np.sqrt(np.diag(np.asarray(fisher.fisher_matrix_inv, dtype=float)))
    for i, name in enumerate(names):
        if name in ranges:
            continue
        half = n_sigma * sigmas[i]
        if not np.isfinite(half) or half <= 0.0:
            raise ValueError(f"non-finite Fisher error for '{name}'")
        ranges[name] = (fiducials[i] - half, fiducials[i] + half)
    return ranges


def posterior_frame(sampler, prior):
    points, log_w, log_l = sampler.posterior()
    data = np.column_stack([points, np.exp(log_w), log_l])
    return pd.DataFrame(data, columns=list(prior.keys) + ["weight", "posterior"])


def n_batch(n_live, n_workers):
    minimum = max(n_live, 4, n_workers)
    return -(-minimum // n_workers) * n_workers


def chain_stats(chain, params):
    """Weighted mean/cov/sigma/ESS of a weighted (nautilus) chain."""
    w = np.asarray(chain["weight"], dtype=float)
    w = w / w.sum()
    x = np.asarray(chain[params], dtype=float)
    mean = w @ x
    cov = np.atleast_2d(np.cov(x.T, aweights=w, ddof=0))
    sig = np.sqrt(np.diag(cov))
    ess = float(1.0 / np.sum(w**2))
    return {"mean": mean, "covariance": cov, "sigma": sig, "ess": ess,
            "corr": cov / np.outer(sig, sig),
            "sigma_frac_err": float(1.0 / np.sqrt(2.0 * ess))}


def fisher_sigma(fisher, params):
    sig = np.asarray(fisher.get_confidence_bounds(), dtype=float)
    names = list(fisher.get_param_names())
    return np.array([sig[names.index(p)] for p in params])

## 1. Fisher forecasts for both probes (symbolic_new)

The same configuration as the chains below; these serve both as contour
overlays and as reference $\sigma$ in the statistics section.

In [ ]:
freepars = {p: 0.01 for p in FREE_PARAMS}

def run_probe(pipeline, label):
    fm = FisherMatrix(
        fiducialpars=FIDUCIAL,
        freepars=freepars,
        options=options_for(f"{label}_symb_demo_"),
        observables=[pipeline],
        cosmoModel="LCDM",
        surveyName="Euclid",
    )
    return fm, fm.compute()


fm_wl, fish_wl = run_probe("WL", "WL")
fm_gcsp, fish_gcsp = run_probe("GCsp", "GCsp")
fish_sum = fish_wl + fish_gcsp
fish_sum.name = "WL + GCsp (symbolic)"

for f in (fish_wl, fish_gcsp, fish_sum):
    print(f.name, "->", f.file_name)


## 2. Nautilus chains

With `SAMPLE_NUISANCES = True` the priors cover the free cosmology plus every
nuisance of the corresponding Fisher (gaussian boxes of
$\pm 5\,\sigma_{\rm marg}$), and the *marginal* Fisher is the like-for-like
reference.  Set `SAMPLE_NUISANCES = False` (top cell) to hold nuisances
fixed - then the *conditional* Fisher applies instead.

In [ ]:
def run_chain(probe, fm, like_cls, cosmo_kw):
    ranges = (
        nuisance_prior_ranges(fm, FREE_PARAMS)
        if SAMPLE_NUISANCES
        else {p: PRIOR_RANGES[p] for p in FREE_PARAMS}
    )
    like = like_cls(**cosmo_kw)
    prior = like.create_nautilus_prior(ranges)
    sampler = like.run_nautilus(
        prior=prior,
        sampler_kwargs={
            "n_live": N_LIVE, "n_networks": 1, "n_batch": n_batch(N_LIVE, N_WORKERS),
            "pool": N_WORKERS,
            "filepath": str(RESULTS / f"nautilus_{probe}_nlive{N_LIVE}_dim{len(ranges)}.hdf5"),
            "resume": True,
        },
        run_kwargs={"n_eff": max(2 * N_LIVE, 20), "verbose": True, "discard_exploration": True},
    )
    frame = posterior_frame(sampler, prior)

    # persist in native fco-readable text format (params + weight + posterior)
    txt = RESULTS / f"chain_{probe}.txt"
    np.savetxt(txt, np.asarray(frame), header=" ".join(frame.columns), comments="")
    frame.to_csv(RESULTS / f"chain_{probe}.csv", index=False)
    # reload through the native loader to prove round-tripping
    fco_frame = fco.load_Nautilus_chains_from_txt(str(txt), param_cols=list(frame.columns[:-2]))
    print(f"{probe}: {len(frame)} weighted draws from {txt}")
    return fco_frame


chain_wl = run_chain("WL", fm_wl, PhotometricLikelihood, {"cosmo_data": fm_wl, "cosmo_theory": fm_wl})
chain_gcsp = run_chain("GCsp", fm_gcsp, SpectroLikelihood, {"cosmoFM_data": fm_gcsp, "cosmoFM_theory": fm_gcsp})

## 3. Combined contours: both chains and all three Fishers

In [ ]:
reference_fisher = marginal_fisher if SAMPLE_NUISANCES else conditional_fisher
ref_wl = reference_fisher(fish_wl, FREE_PARAMS)
ref_gcsp = reference_fisher(fish_gcsp, FREE_PARAMS)
ref_sum = reference_fisher(fish_sum, FREE_PARAMS)

truth = {p: FIDUCIAL[p] for p in FREE_PARAMS}

tri = fco.make_triangle_plot(
    fishers=[ref_wl, ref_gcsp, ref_sum],
    fisher_labels=[f"WL Fisher ({'marg' if SAMPLE_NUISANCES else 'cond'})",
                   f"GCsp Fisher ({'marg' if SAMPLE_NUISANCES else 'cond'})",
                   "WL+GCsp Fisher"],
    chains=[chain_wl, chain_gcsp],
    chain_labels=["WL Nautilus", "GCsp Nautilus"],
    params=FREE_PARAMS,
    truth_values=truth,
    smooth=3, bins=30,
    shade_fisher=False, shade_chains=True,
    savefile=str(PLOTS / "nautilus_gcsp_wl_contours.png"),
)

## 4. Chain statistics (`fishconsumer` / ChainConsumer)

`fco.plot_chain_summary` renders the ChainConsumer errorbar/summary plot
(68 % / 95 % intervals per parameter for every chain).  The table below adds
the weighted-chain diagnostics: effective sample size (ESS), chain mean vs
fiducial **bias** (in units of the chain $\sigma$, consistent with 0 up to
$1/\sqrt{\rm ESS}$), chain/Fisher $\sigma$ ratio, and the Pearson correlation
of the two sampled parameters.

In [ ]:
fig = fco.plot_chain_summary(
    chains=[chain_wl, chain_gcsp],
    chain_names=["WL", "GCsp"],
    truth_values=truth,
    output_file=str(PLOTS / "nautilus_chain_summary.png"),
)

rows = []
for probe, chain, ref in (("WL", chain_wl, ref_wl), ("GCsp", chain_gcsp, ref_gcsp)):
    st = chain_stats(chain, FREE_PARAMS)
    sf = fisher_sigma(ref, FREE_PARAMS)
    bias = (st["mean"] - np.array([FIDUCIAL[p] for p in FREE_PARAMS])) / st["sigma"]
    for i, p in enumerate(FREE_PARAMS):
        rows.append({
            "chain": probe, "param": p,
            "mean": st["mean"][i], "sigma_chain": st["sigma"][i],
            "sigma_fisher": sf[i], "sigma_ratio": st["sigma"][i] / sf[i],
            "bias_sigma": bias[i], "ESS": st["ess"],
            "sigma_MC_err_%": 100 * st["sigma_frac_err"],
        })
    print(f"{probe}: corr(chain) = {st['corr'][0, 1]:+.4f}   ESS = {st['ess']:.0f}")

df_stats = pd.DataFrame(rows).set_index(["chain", "param"])
display(df_stats)

In [ ]:
# Fisher-correlation comparison of the two sampled parameters
def corr_of(fisher, params):
    cov = np.asarray(fisher.fisher_matrix_inv, dtype=float)
    idx = [list(fisher.param_names).index(p) for p in params]
    sub = cov[np.ix_(idx, idx)]
    d = np.sqrt(np.diag(sub))
    return sub / np.outer(d, d)

for probe, chain, ref in (("WL", chain_wl, ref_wl), ("GCsp", chain_gcsp, ref_gcsp)):
    c_ch = chain_stats(chain, FREE_PARAMS)["corr"][0, 1]
    c_fi = corr_of(ref, FREE_PARAMS)[0, 1]
    print(f"{probe:5s} correlation: chain {c_ch:+.4f}  vs  Fisher {c_fi:+.4f}  "
          f"(difference {abs(c_ch - c_fi):.4f})")

## 5. Fisher-level statistic: KL information gain (bits)

`cfop.information_gain(F1, F2, prior)` measures how many bits of information
one Fisher forecast carries relative to another (added to a flat prior); the
statistical-average term of the Gaussian Fisher approximation is included.

In [ ]:
from cosmicfishpie.analysis.fisher_matrix import fisher_matrix as FMcls

def zero_prior_like(f):
    Fz = 0.0 * np.asarray(f.get_fisher_matrix(), dtype=float)
    return FMcls(fisher_matrix=Fz, param_names=f.get_param_names(),
                 param_names_latex=f.get_param_names_latex(),
                 fiducial=f.get_param_fiducial())

info_gcsp_vs_wl = cfop.information_gain(fish_gcsp, fish_wl, zero_prior_like(fish_gcsp), stat=True)
print(f"information gain GCsp vs WL: {float(info_gcsp_vs_wl):.3f} bits")

## Recap and knobs

* `FREE_PARAMS` / `SAMPLE_NUISANCES` / `N_LIVE` / `N_WORKERS` at the top control the run.
* Chains resume from the `.hdf5` checkpoints in `notebook results/` - delete them
  to restart from scratch (resume silently reuses the exploration bounds, so also
  delete them after changing `N_LIVE` or the dimensionality).
* All plots land in `notebooks/plots/` (git-ignored); chains, checkpoints and the
  two symbolic Fishers in `notebooks/results/` can be committed.
* Raise `N_LIVE` well above 50 (and `n_eff` accordingly) before quoting numbers -
  this notebook is a speed/recall demo of `fco` chain statistics, not a production
  chain.